In [1]:
import json
import os

In [94]:
scores_dir = 'qa_decode_run2'
scores = os.listdir(scores_dir)
scores.sort(key=lambda x: x[slice(x.find("_en") + 1, x.find("_en") + 5)], reverse=True)

In [95]:
scores

['none_enzh_ntrex-128_results_scores.json',
 'natural_token_enzh_ntrex-128_results_scores.json',
 'natural_sequence_enzh_ntrex-128_results_scores.json',
 'natural_segment_enzh_ntrex-128_results_scores.json',
 'comet_enzh_ntrex-128_results_scores.json',
 'none_enms_ntrex-128_results_scores.json',
 'natural_token_enms_ntrex-128_results_scores.json',
 'natural_sequence_enms_ntrex-128_results_scores.json',
 'natural_segment_enms_ntrex-128_results_scores.json',
 'comet_enms_ntrex-128_results_scores.json',
 'none_enko_ntrex-128_results_scores.json',
 'natural_token_enko_ntrex-128_results_scores.json',
 'natural_sequence_enko_ntrex-128_results_scores.json',
 'natural_segment_enko_ntrex-128_results_scores.json',
 'comet_enko_ntrex-128_results_scores.json']

In [96]:
for i in range(0, len(scores), 5):
    scores[i+1:i+3] = scores[i+1:i+3][::-1]

scores

['none_enzh_ntrex-128_results_scores.json',
 'natural_sequence_enzh_ntrex-128_results_scores.json',
 'natural_token_enzh_ntrex-128_results_scores.json',
 'natural_segment_enzh_ntrex-128_results_scores.json',
 'comet_enzh_ntrex-128_results_scores.json',
 'none_enms_ntrex-128_results_scores.json',
 'natural_sequence_enms_ntrex-128_results_scores.json',
 'natural_token_enms_ntrex-128_results_scores.json',
 'natural_segment_enms_ntrex-128_results_scores.json',
 'comet_enms_ntrex-128_results_scores.json',
 'none_enko_ntrex-128_results_scores.json',
 'natural_sequence_enko_ntrex-128_results_scores.json',
 'natural_token_enko_ntrex-128_results_scores.json',
 'natural_segment_enko_ntrex-128_results_scores.json',
 'comet_enko_ntrex-128_results_scores.json']

In [97]:
code2name = {
    "enzh": "Chinese",
    "enfr": "French",
    "ende": "German",
    "enms": "Malay",
    "enko": "Korean"
}

In [98]:
metrics = ['COMET', 'XCOMET', 'CometKiwi', 'spBLEU', 'METEOR', 'Naturalness']

In [ ]:
def results_table():
    ind = 0
    langs = set()
    print("\\toprule")
    for i, score_file in enumerate(scores):
        with open(f"{scores_dir}/{score_file}", "r") as file:
            results = json.load(file)

        teval_level = score_file.split('_')[1] if "natural" in score_file else score_file.split('_')[0]
        idx = score_file.index("_en") + 1
        lang = slice(idx, idx+4)
        lang_pair = score_file[lang]

        if ind == 0:
            head = f"Model\t&\t\\multicolumn{{3}}{{c|}}{{Semantic-based}}\t&\t\\multicolumn{{2}}{{c|}}{{N-gram-based}}\t&\t\\multicolumn{{3}}{{c|}}{{Naturalness}}\t&\t\\multirow{{2}}{{*}}{{Average}}\t\\\\"
            print(head)
            header = "\\quad + QE$_{{~\\text{{granularity}}}}$\t"
            for metric in metrics:
                if isinstance(results[metric], dict):
                    for met in results[metric]:
                        header += f"&\t{met.capitalize()}\t"
                elif not metric in head:
                    header += f"&\t{metric}\t"
                else:
                    header += "&\t"
            header += '&\t\\\\'
            print(header)
        if not lang_pair in langs:
            print('\\midrule')
            print(f"\\multicolumn{{10}}{{c}}{{{lang_pair}}}\t\\\\")
            print(f"\\midrule")
            langs.add(lang_pair)
        if teval_level == "none":
            row = "Qwen3-4B\t"
        elif teval_level == "comet":
            row = f"\\quad + XCOMET\t"
        else:
            row = f"\\quad + Nat.$_\\text{{{teval_level[:3]}}}$\t"
        
        total = 0
        for metric in metrics:
            if isinstance(results[metric], dict):
                for _, score in results[metric].items():
                    total += score
                    row += f"&\t{score:.2f}\t"
            else:
                score = results[metric]
                if score < 1:
                    score *= 100
                total += score
                row += f"&\t{score:.2f}\t"
        avg = total / len(metrics)
        row += f'&\t{avg:.2f}\t\\\\'

        print(row)
        ind += 1
    print("\\bottomrule")

In [ ]:
results_table()

\toprule
Model	&	\multicolumn{3}{c|}{Semantic-based}	&	\multicolumn{2}{c|}{N-gram-based}	&	\multicolumn{3}{c|}{Naturalness}	&	\multirow{2}{*}{Average}	\\
\quad + QE$_{{~\text{{granularity}}}}$	&	COMET	&	XCOMET	&	CometKiwi	&	spBLEU	&	METEOR	&	Token	&	Segment	&	Sequence	&	\\
\midrule
\multicolumn{10}{c}{enzh}	\\
\midrule
Qwen3-4B	&	85.68	&	80.34	&	82.88	&	19.58	&	5.41	&	47.85	&	48.63	&	47.86	&	69.71	\\
\quad + Nat.$_\text{seq}$	&	85.46	&	78.95	&	81.60	&	19.58	&	4.71	&	52.33	&	50.05	&	52.34	&	70.83	\\
\quad + Nat.$_\text{tok}$	&	85.50	&	78.93	&	81.47	&	19.58	&	4.70	&	52.37	&	49.99	&	52.38	&	70.82	\\
\quad + Nat.$_\text{seg}$	&	85.57	&	79.23	&	81.71	&	19.58	&	5.00	&	51.70	&	50.26	&	51.70	&	70.79	\\
\quad + XCOMET	&	85.91	&	81.84	&	82.64	&	15.99	&	5.42	&	48.24	&	48.64	&	48.24	&	69.49	\\
\midrule
\multicolumn{10}{c}{enms}	\\
\midrule
Qwen3-4B	&	83.09	&	76.58	&	77.49	&	15.91	&	49.89	&	48.15	&	48.89	&	48.15	&	74.69	\\
\quad + Nat.$_\text{seq}$	&	82.09	&	75.60	&	77.31	&	15.51	&	48.72	&	51.09	&	